# 02 · City B Viana do Castelo: stream profile and prequential experiment

**PA04 · Textile storage facility (humid vs. dry regimes) · T4 Humid vs Dry Regime**
**Author:** Victoriya Yurevna (20230796), Student B · **Primary city:** Viana do Castelo

* **Business context:** textile storage facility (humid vs. dry regimes)
* **Locations:** Guimarães vs Viana do Castelo
* **Question:** How do humid and dry regimes differ between candidate warehouse locations?

Individual notebook. Prerequisite: `00_shared_protocol` has been run (shared protocol checks and records).
The setup cell loads the shared protocol; the guide below shows how to use the modules in `src/`.

<div class="alert alert-block alert-warning" style="background-color:#ffe94d; border:4px solid #ff8c00; border-radius:6px; padding:14px 18px; color:#000000;">

> ⚠️ **TEMPORARY NOTE: DELETE THIS CELL BEFORE SUBMISSION.**
> Working guidance for the two of us while we experiment. It is not part of the deliverable.

## Experimenting without breaking the shared protocol

Both city notebooks must produce **official results under the identical protocol**, so that `03_shared_comparison` compares like with like (Ch. 3.1). Experiments are welcome on top of that, as long as they never change or replace the official run.

### Fixed by the shared protocol: never change these here
* Data: period, cities, variables, source (`config.yaml → data`, `cities`).
* Warm-up (first 12 months) and τ_H (Q0.50 of the pooled warm-up = 82 %, frozen).
* Target T4: y = 1 if RH_t ≥ τ_H.
* Strict chronological order and PREDICT → EVALUATE → LEARN (`evaluate.run_prequential`).
* The official run: the 5 mandatory models, Representations A and B, and the settings in `config.yaml`.
* Leakage rules: RH at t is never a predictor, and nothing is fitted on evaluation data.

### Open for experiments (tested with the current modules)

| Experiment | How |
|---|---|
| Other window sizes, rolling statistics of other variables | `my_cfg["features"]["windows"] = [3, 12, 48]`; `my_cfg["features"]["rolling_variables"] += ["temperature_2m"]`, then `make_streams(prepared, my_cfg)[CITY_KEY]`. `from features import check_no_leakage; check_no_leakage(prepared["raw"][CITY_KEY], TAU_H, my_cfg)` also covers these features. |
| Your own features (e.g. hour of day, dew point at t−1, rain over the last 24 h) | Add them to each event after `make_streams`: `e["x"]["hour_sin"] = ...`. Use only values up to t−1, or calendar values (known in advance). These are not checked by `check_no_leakage` and not scaled, so if you scale them, fit the scaling on the warm-up only. |
| Hyper-parameters | `my_cfg["models"]["hoeffding"]["grace_period"] = 200`, `my_cfg["models"]["ensemble_size"]`, `arf_lambda`, `logistic_lr`, … then `evaluate_models(streams, make_models, my_cfg, CITY_KEY)`. |
| Extra models next to the mandatory five | Any River classifier: `from river import neighbors, tree`, then `extra = lambda c: {"Hoeffding Adaptive Tree": tree.HoeffdingAdaptiveTreeClassifier(seed=c["protocol"]["seed"]), "kNN": neighbors.KNNClassifier()}`, then `evaluate_models(streams, extra, cfg, CITY_KEY)`. |
| Another "recent performance" window | `my_cfg["protocol"]["evaluation_window"] = 168` |
| Original descriptive analyses (Ch. 3.2) | Fully open, on `profile["df"]`. |
| Business-risk views (descriptive only) | e.g. `profiling.episodes(df.assign(y=(df["relative_humidity_2m"] >= 90).astype(float)))`. Never redefine the T4 target. |

**Not possible yet: scores per season or per month, or errors during long humid episodes.** `run_prequential` does not keep each individual prediction. Do not rewrite the evaluation loop in this notebook. Instead, we can add an option to `evaluate.py` that is off by default (so the official run is unchanged), and only after we agree on it together.

### Rules that preserve the shared structure
1. **Always experiment on a copy:** `import copy; my_cfg = copy.deepcopy(cfg)`. Never edit `cfg` in place.
2. **Never call `save_results` or `profiling.save_profile` with a modified configuration.** These write the official files that notebook 03 reads.
3. **Name experiment figures `<city_key>_exp_<name>`**, so they can never overwrite an official figure.
4. **Never edit `config.yaml` or `src/` alone.** Both change the other city's results too. Propose the change, agree, then both re-run.
5. **Keep experiments in their own clearly titled section**, separate from the official results, and report them as additional analysis.
6. **Promoting an experiment to the official run is a joint decision:** change `config.yaml` together, then re-run `00`, `01`, `02` and `03`.

</div>

## Setup

In [ ]:
CITY_KEY = "viana_do_castelo"              # the only line that differs between the two city notebooks

import sys
from pathlib import Path

# Make src/ importable whatever folder Jupyter was started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

import plots
import profiling
from data import prepare_frames
from evaluate import evaluate_models, save_results
from features import make_streams
from models import make_models
from utils import load_config, output_dir

plots.set_style()
cfg = load_config()
prepared = prepare_frames(cfg)                 # shared protocol: same tau_H, warm-up and target as notebook 00
CITY = next(c for c in cfg["cities"] if c["key"] == CITY_KEY)
TAU_H = prepared["tau"]
df = prepared["frames"][CITY_KEY]              # this city's stream: timestamps, 4 variables, quality_flag, phase, y
print(f"{CITY['role']}: {CITY['name']}  |  {len(df)} hours  |  tau_H = {TAU_H:.2f} %")

## How to use the shared modules

The setup cell gives you `cfg` (`config.yaml`), `prepared` (output of `data.prepare_frames`: `frames`, `tau`, `n_warm`, `warmup_end`, `evaluation_start`, `payloads`), `TAU_H` and `df`, this city's stream.
Never recompute τ_H or set a protocol value here: everything comes from `config.yaml` through `src/`, so both cities stay comparable (Ch. 3.1).

### 1. Stream profile: `src/profiling.py`

```python
profile = profiling.profile_city(df, TAU_H, cfg)     # dict of tables, same logic for both cities
profile["class_by_season"]                           # any key below
```

| Key | Content | Ch. 9 item |
|---|---|---|
| `quality`, `special` | missing / flagged values, constant runs, RH = 100 %, calm hours, rain hours | 1 |
| `describe`, `by_phase`, `by_season`, `daily` | descriptive statistics, warm-up vs evaluation, seasons, daily aggregates | 2–3 |
| `class_by_phase`, `class_by_year`, `class_by_season` | humid / dry shares | 4 |
| `rolling`, `rolling_by_season` | 6 h / 24 h rolling mean and std of RH, 7-day humid share | 5 |
| `extremes`, `top_extremes` (+ `profiling.extreme_days(profile["daily"])`) | tails and most extreme hours | 6 |
| `episodes`, `persistence`, `humid_episodes_by_season` | how long humid and dry regimes last | original analysis |
| `hour_month`, `daily_cycle` | humid share by month × local hour | original analysis |
| `transitions`, `persistence_reference` | P(y_t given y_t−1), and the persistence rule ŷ_t = y_t−1 scored on the evaluation stream (the bar the models must clear) | original analysis |
| `key_numbers` | one summary series used by `03_shared_comparison` | — |
| `df` | the stream with `local_time`, `hour`, `month`, `year`, `season` added | — |

Each function also works on its own, e.g. `profiling.episodes(df)` or `profiling.persistence_reference(df)`. Add your own analyses on top of `profile["df"]`.

Save the tables for the report and for notebook 03:
```python
profiling.save_profile(profile, CITY_KEY, output_dir(cfg, "profiles_dir"))   # results/profiles/<city>_*.csv
```

### 2. Figures: `src/plots.py`

```python
fig = plots.daily_evolution(profile, prepared, cfg, CITY["name"])
plots.save(fig, cfg, f"{CITY_KEY}_01_daily_evolution")          # results/figures/<city>_NN_<name>.png
plt.show()
```
Available figures (each returns a matplotlib figure):

| Call | Figure |
|---|---|
| `plots.daily_evolution(profile, prepared, cfg, CITY["name"])` | daily mean (total for rain) of the 4 variables |
| `plots.rh_by_month(profile, prepared, cfg, CITY["name"])` | hourly RH by calendar month, with τ_H |
| `plots.humid_share_month_year(profile, CITY["name"])` | humid share by month, one bar per year |
| `plots.rolling_overview(profile, prepared, CITY["name"])` | 24 h rolling mean / std of RH, 7-day humid share |
| `plots.rolling_zoom(profile, prepared, cfg, CITY["name"], start="2024-07-01", days=14)` | 6 h vs 24 h windows on a short excerpt |
| `plots.episode_durations(profile, CITY["name"])` | duration of humid and dry episodes |
| `plots.hour_month_heatmap(profile, CITY["name"])` | humid share by month × local hour |
| `plots.recent_performance(history, cfg, representation="B")` | sliding-window Macro-F1 of each model over time |

They share one style and fixed color roles (humid = blue, dry = orange). Name your files `<city_key>_NN_<name>` so the other notebooks can find them.

### 3. Prequential experiment: `src/features.py`, `src/models.py`, `src/evaluate.py`

```python
streams = make_streams(prepared, cfg)[CITY_KEY]       # {"A": {"warm": [...], "eval": [...]}, "B": {...}}
metrics, history = evaluate_models(streams, make_models, cfg, CITY_KEY)   # 2 representations x 5 models, ~15 s
save_results(metrics, history, CITY_KEY, cfg)         # results/metrics_<city>.csv, history_<city>.csv (official run only)

fig = plots.recent_performance(history, cfg, representation="B")         # sliding-window Macro-F1 over time
```
* Representation A = values at t−1. Representation B = A + delta, 6 h / 24 h rolling mean and std of RH, humid hours in the previous 24 h (all ending at t−1).
* `metrics` columns: `accuracy`, `macro_f1`, `precision`, `recall`, `f1_event` (positive class = humid), `tn`, `fp`, `fn`, `tp`, `pred_ms_per_event`, `update_ms_per_event`.
* Compare with `profile["persistence_reference"]`.

### 4. Individual experiments

Work on a **copy** of the configuration, so the official results stay comparable with the other city:
```python
import copy
my_cfg = copy.deepcopy(cfg)
my_cfg["features"]["windows"] = [3, 12]              # example
my_streams = make_streams(prepared, my_cfg)[CITY_KEY]
my_metrics, my_history = evaluate_models(my_streams, make_models, my_cfg, CITY_KEY)
```
Never call `save_results` with a modified configuration. If a change is clearly better, agree on it together, change `config.yaml`, and both re-run your notebooks.

### What this notebook should cover (assignment)

* Stream characterisation of the city, Ch. 9 items 1–7: data quality, descriptive statistics, temporal evolution, class distribution, rolling / window behaviour, unusual observations, business interpretation.
* At least two original analyses (Ch. 3.2).
* The prequential experiment on this city and its interpretation (Ch. 8, 10–12).
* 📝 Interpretation written by yourself (AI policy, Ch. 16).
* Outputs saved with `profiling.save_profile` and `save_results`, so `03_shared_comparison` can pair both cities.